# Python Regular Expressions (Regex) — Pattern Matching & String Processing

> **Topic:** Regular Expressions (`re` module) | **Folder:** Data Structures & Algorithms

A **Regular Expression (Regex)** is a powerful sequence of characters that forms a search pattern.
Regex is used for text search, string parsing, data validation, lexical analysis (tokenization),
and data extraction in data structures and algorithms.

Python provides built-in support for regular expressions via the **`re` module**.

---

## Table of Contents
1. [Introduction to Python's `re` Module & Raw Strings](#1.-Introduction-to-Python's-`re`-Module-&-Raw-Strings)
2. [Core Search Functions (`search`, `match`, `fullmatch`, `findall`, `finditer`)](#2.-Core-Search-Functions-(search,-match,-fullmatch,-findall,-finditer))
3. [Metacharacters & Character Classes](#3.-Metacharacters-&-Character-Classes)
4. [Quantifiers: Greedy vs. Non-Greedy (Lazy) Matching](#4.-Quantifiers:-Greedy-vs.-Non-Greedy-(Lazy)-Matching)
5. [Grouping, Capturing & Named Groups](#5.-Grouping,-Capturing-&-Named-Groups)
6. [Lookahead & Lookbehind Assertions (Zero-Width Checks)](#6.-Lookahead-&-Lookbehind-Assertions-(Zero-Width-Checks))
7. [String Substitution & Splitting (`re.sub`, `re.split`)](#7.-String-Substitution-&-Splitting-(re.sub,-re.split))
8. [Pattern Compilation & Regex Flags (`re.compile`, `re.IGNORECASE`, `re.VERBOSE`)](#8.-Pattern-Compilation-&-Regex-Flags-(re.compile,-re.IGNORECASE,-re.VERBOSE))
9. [Practical Algorithmic Applications (Tokenization, Log Parsing, Validation)](#9.-Practical-Algorithmic-Applications-(Tokenization,-Log-Parsing,-Validation))
10. [Performance, Catastrophic Backtracking & Best Practices](#10.-Performance,-Catastrophic-Backtracking-&-Best-Practices)
11. [Quick Reference Card](#11.-Quick-Reference-Card)


---
## 1. Introduction to Python's `re` Module & Raw Strings

Always use **Raw Strings** (`r"pattern"`) when writing regular expressions in Python.
Raw strings prevent Python's string parser from interpreting backslashes `\` as standard string escape sequences
(e.g., `\n`, `\t`), passing raw backslashes directly to the Regex engine.


In [ ]:
# Raw string comparison
normal_str = "\n"    # Represents a newline character
raw_str    = r"\n"   # Represents literal backslash followed by 'n'

print(f"len(normal_str): {len(normal_str)}")  # 1
print(f"len(raw_str)   : {len(raw_str)}")     # 2


---
## 2. Core Search Functions (`search`, `match`, `fullmatch`, `findall`, `finditer`)

| Function | Scope | Return Value |
|----------|-------|--------------|
| `re.search(pattern, string)` | Searches **anywhere** in string | `Match` object for 1st match, or `None` |
| `re.match(pattern, string)` | Checks match at **start** of string | `Match` object, or `None` |
| `re.fullmatch(pattern, string)` | Checks if **entire string** matches | `Match` object, or `None` |
| `re.findall(pattern, string)` | Searches **entire string** | List of matching strings |
| `re.finditer(pattern, string)` | Searches **entire string** | Iterator of `Match` objects |


In [ ]:
import re

text = "Order #1024 placed on 2026-08-13. Order #1025 placed on 2026-08-14."

# 1. re.search() -> First occurrence
m_search = re.search(r"\d{4}", text)
if m_search:
    print(f"re.search() match : '{m_search.group()}' at span {m_search.span()}")

# 2. re.match() -> Only matches at index 0
m_match = re.match(r"Order", text)
print(f"re.match() at start: {bool(m_match)}")

# 3. re.findall() -> List of strings
all_orders = re.findall(r"#\d{4}", text)
print(f"re.findall() matches: {all_orders}")

# 4. re.finditer() -> Iterator of Match objects (provides span and context)
print("\nre.finditer() detailed matches:")
for match in re.finditer(r"\d{4}-\d{2}-\d{2}", text):
    print(f"  Found date '{match.group()}' from index {match.start()} to {match.end()}")


---
## 3. Metacharacters & Character Classes

### Character Classes
- `\d` = Digit `[0-9]`  |  `\D` = Non-digit `[^0-9]`
- `\w` = Word character `[a-zA-Z0-9_]`  |  `\W` = Non-word character
- `\s` = Whitespace `[ \t\n\r\f\v]`  |  `\S` = Non-whitespace
- `.` = Any character except newline `\n`
- `^` = Start of string  |  `$` = End of string


In [ ]:
sample = "User ID: admin_99, Code: X-501!"

digits = re.findall(r"\d+", sample)
words  = re.findall(r"\w+", sample)
custom = re.findall(r"[A-Z]-\d+", sample)  # Custom range [A-Z] followed by hyphen and digits

print(f"Digits found: {digits}")
print(f"Words found : {words}")
print(f"Custom code : {custom}")


---
## 4. Quantifiers: Greedy vs. Non-Greedy (Lazy) Matching

| Quantifier | Meaning | Matching Mode |
|------------|---------|---------------|
| `*` | 0 or more | **Greedy** (matches as much as possible) |
| `+` | 1 or more | **Greedy** |
| `?` | 0 or 1 | **Greedy** |
| `*?` | 0 or more | **Lazy** (matches as little as possible) |
| `+?` | 1 or more | **Lazy** |
| `{n,m}` | Between n and m times | **Greedy** |
| `{n,m}?` | Between n and m times | **Lazy** |


In [ ]:
html = "<div>First Block</div><div>Second Block</div>"

# Greedy: matches from 1st <div> to the VERY LAST </div>
greedy_match = re.search(r"<div>.*</div>", html).group()

# Lazy: matches <div> up to the NEAREST </div>
lazy_match = re.search(r"<div>.*?</div>", html).group()
all_lazy = re.findall(r"<div>.*?</div>", html)

print(f"Greedy match : {greedy_match}")
print(f"Lazy match   : {lazy_match}")
print(f"All Lazy tags: {all_lazy}")


---
## 5. Grouping, Capturing & Named Groups

Parentheses `(...)` define capturing groups.  
Named groups `(?P<name>...)` allow accessing captured sub-patterns by name rather than index.


In [ ]:
log_line = "2026-08-13 14:30:45 [ERROR] Database connection failed: Host 192.168.1.50 timeout"

# Named capture groups
log_pattern = r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) \[(?P<level>\w+)\] (?P<message>.*)"

match = re.search(log_pattern, log_line)
if match:
    print("Extracted Log Fields:")
    print(f"  Date   : {match.group('date')}")
    print(f"  Time   : {match.group('time')}")
    print(f"  Level  : {match.group('level')}")
    print(f"  Message: {match.group('message')}")
    print("\nGroup Dictionary:", match.groupdict())


---
## 6. Lookahead & Lookbehind Assertions (Zero-Width Checks)

Lookaround assertions test conditions **without consuming characters** in the matching string.

| Syntax | Type | Meaning |
|--------|------|---------|
| `(?=pattern)` | Positive Lookahead | Followed by pattern |
| `(?!pattern)` | Negative Lookahead | NOT followed by pattern |
| `(?<=pattern)` | Positive Lookbehind | Preceded by pattern |
| `(?<!pattern)` | Negative Lookbehind | NOT preceded by pattern |


In [ ]:
text = "Prices: $100, €250, ¥3000, 400USD"

# Positive Lookbehind: Find numbers preceded by '$'
usd_prices = re.findall(r"(?<=\$)\d+", text)
print(f"USD prices (preceded by $): {usd_prices}")

# Positive Lookahead: Find numbers followed by 'USD'
usd_code = re.findall(r"\d+(?=USD)", text)
print(f"Amounts followed by 'USD' : {usd_code}")

# Password Strength Validator (Positive Lookahead chain):
# Requires at least 1 uppercase, 1 lowercase, 1 digit, min 8 chars
pwd_pattern = r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d)[a-zA-Z\d]{8,}$"

passwords = ["weak", "Password123", "no_digits_Here", "12345678"]
for p in passwords:
    valid = bool(re.match(pwd_pattern, p))
    print(f"  Password '{p:<15}' -> Valid: {valid}")


---
## 7. String Substitution & Splitting (`re.sub`, `re.split`)

`re.sub(pattern, replacement, string)` replaces matches with a target string or a **callback function**.
`re.split(pattern, string)` splits a string on any complex regex delimiter.


In [ ]:
# 1. Redacting sensitive data (PII Masking)
data = "Contact Alice at alice@example.com or Bob at bob@company.org"
masked_data = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", "[REDACTED]", data)
print(f"Masked text: {masked_data}")

# 2. Using a callback function in re.sub()
# Convert all prices in text from USD to EUR (e.g. rate 0.92)
prices_text = "Items cost $50 and $120."

def convert_usd_to_eur(match):
    val = float(match.group(1))
    return f"€{val * 0.92:.2f}"

converted = re.sub(r"\$(\d+)", convert_usd_to_eur, prices_text)
print(f"Converted text: {converted}")

# 3. Complex splitting on multiple delimiters (commas, semicolons, whitespace)
raw_tokens = "apple, banana; cherry   date,elderberry"
clean_tokens = re.split(r"[;,\s]+", raw_tokens)
print(f"Split tokens  : {clean_tokens}")


---
## 8. Pattern Compilation & Regex Flags (`re.compile`)

Compiling patterns with `re.compile()` pre-builds the internal regex state machine,
improving performance when the pattern is reused inside loops.

### Common Flags
- `re.IGNORECASE` (`re.I`): Case-insensitive matching.
- `re.MULTILINE` (`re.M`): `^` and `$` match start/end of **each line**.
- `re.DOTALL` (`re.S`): `.` matches **all** characters including newlines `\n`.
- `re.VERBOSE` (`re.X`): Allows formatted, commented, multi-line regexes.


In [ ]:
# Pre-compiling with VERBOSE flag for readability
email_regex = re.compile(r"""
    ^                           # Start of string
    (?P<username>[a-zA-Z0-9._%+-]+) # Username
    @                           # @ symbol
    (?P<domain>[a-zA-Z0-9.-]+)  # Domain name
    \.                         # Literal dot
    (?P<tld>[a-zA-Z]{2,})       # Top-level domain
    $                           # End of string
""", re.VERBOSE | re.IGNORECASE)

m = email_regex.match("Dev.User@PythonGym.org")
if m:
    print("Compiled Verbose Regex Match:", m.groupdict())


---
## 9. Practical Algorithmic Applications (Tokenization, Parsing)

Tokenization is a core component of compilers and natural language processing (NLP).
Here is a simple **Lexical Analyzer (Tokenizer)** built using Regex iterators.


In [ ]:
# Simple Arithmetic Expression Tokenizer
TOKEN_SPEC = [
    ('NUMBER',   r'\d+(\.\d+)?'), # Integer or decimal number
    ('ASSIGN',   r'='),             # Assignment operator
    ('END',      r';'),             # Statement terminator
    ('ID',       r'[A-Za-z_]\w*'), # Identifiers
    ('OP',       r'[+\-*/]'),       # Arithmetic operators
    ('SKIP',     r'[ \t]+'),        # Skip spaces and tabs
    ('MISMATCH', r'.'),             # Any other character
]

tok_regex = '|'.join(f'(?P<{pair[0]}>{pair[1]})' for pair in TOKEN_SPEC)

code_snippet = "x = 42 + y * 3.14;"
tokens = []

for mo in re.finditer(tok_regex, code_snippet):
    kind = mo.lastgroup
    value = mo.group()
    if kind == 'SKIP': continue
    elif kind == 'MISMATCH': raise RuntimeError(f"Unexpected char '{value}'")
    tokens.append((kind, value))

print(f"Code Snippet: '{code_snippet}'")
print("Tokens generated:")
for t in tokens: print(f"  {t[0]:<10} -> {t[1]}")


---
## 10. Performance & Catastrophic Backtracking

### Catastrophic Backtracking (ReDoS)
Poorly designed patterns with nested quantifiers (e.g., `(a+)+b`) evaluated on non-matching inputs
cause the NFA engine to explore **exponential $O(2^n)$ combinations**, hanging the processor!


In [ ]:
# Benchmark: Regex vs Native String Methods
import timeit

sample_text = "https://python.org"

# Checking prefix with Regex vs str.startswith()
t_regex  = timeit.timeit(lambda: bool(re.match(r"^https://", sample_text)), number=100_000)
t_native = timeit.timeit(lambda: sample_text.startswith("https://"), number=100_000)

print(f"Regex match time : {t_regex:.4f} seconds")
print(f"Native startswith: {t_native:.4f} seconds")
print(f"Native is ~{t_regex / t_native:.1f}x faster for simple string checks!")


---
## 11. Quick Reference Card


In [ ]:
# ==================================================================
# PYTHON REGEX – QUICK REFERENCE
# ==================================================================
import re

s = "User: Alice (ID: 101), Email: alice@test.com"

# Search & Findall
print("ID     :", re.search(r"ID: (\d+)", s).group(1))
print("Words  :", re.findall(r"\b[A-Za-z]+\b", s))

# Sub & Split
print("Masked :", re.sub(r"\d+", "***", s))
print("Split  :", re.split(r"[:,\s]+", s)[:4])

# Lookaround
print("Lookbehind ID:", re.findall(r"(?<=ID: )\d+", s))


---
## Summary

| Tool / Pattern | Usage | Best Used For |
|----------------|-------|---------------|
| `re.search()` | Find 1st match anywhere | Single item extraction |
| `re.findall()` | Return all matches as list | Bulk extraction |
| `re.finditer()` | Return iterator of `Match` objects | Complex extraction with spans & groups |
| `re.sub()` | Search and replace | Data sanitization & masking |
| `(?P<name>...)` | Named capture groups | Structured parsing (logs, dates) |
| `(?=...)` / `(?<=...)` | Zero-width lookarounds | Conditional validation without consuming chars |

---
*Next up: **Object Oriented Programming (Classes & Inheritance)***
